## Part 3: Summary Statistics
Summary statistics for the important variables in our cleaned data (`games_clean.csv` and `reviews_clean.csv`).

In [1]:
import pandas as pd

games = pd.read_csv("games_clean.csv", parse_dates=["release_date"])
reviews = pd.read_csv("reviews_clean.csv")

print(len(games), "games")
print(len(reviews), "reviews")

121 games
11845 reviews


### Step 1: Create the variables we need
* `total_reviews` = positive + negative reviews
* `pct_positive` = percent of reviews that recommend the game (this is the same number Steam uses for its rating)
* `release_year` = year the game came out
* `playtime_hours` = playtime converted from minutes to hours
* `word_count` = number of words in the review (the "Unreviewed" placeholders count as 0)

In [2]:
games["total_reviews"] = games["total_positive"] + games["total_negative"]
games["pct_positive"] = games["total_positive"] / games["total_reviews"] * 100
games["release_year"] = games["release_date"].dt.year

reviews["playtime_hours"] = reviews["playtime_at_review"] / 60
reviews["word_count"] = reviews["review_text"].str.split().str.len()
reviews.loc[reviews["review_text"] == "Unreviewed", "word_count"] = 0

### Step 2: Summary statistics for the games
`.describe()` gives the count, mean (average), standard deviation, minimum, quartiles, and maximum for each column.

In [3]:
games[["price", "pct_positive", "total_reviews", "owners_avg", "release_year"]].describe().round(2)

,price,pct_positive,total_reviews,owners_avg,release_year
count,121.00,121.00,121.00,121.00,121.00
mean,14.34,75.88,8183.61,590578.51,2021.98
std,8.43,13.59,67832.52,3266001.13,4.46
min,0.00,32.09,54.00,10000.00,2005.00
25%,7.99,67.16,194.00,10000.00,2018.00
50%,14.88,75.98,412.00,35000.00,2024.00
75%,19.99,86.86,1469.00,350000.00,2026.00
max,39.99,98.03,746768.00,35000000.00,2026.00


### Step 3: Summary statistics for the reviews

In [4]:
reviews[["playtime_hours", "word_count", "votes_helpful"]].describe().round(2)

,playtime_hours,word_count,votes_helpful
count,11845.00,11845.00,11845.00
mean,51.06,73.87,3.52
std,483.02,138.50,12.33
min,0.08,0.00,0.00
25%,2.45,9.00,0.00
50%,6.20,27.00,1.00
75%,18.12,77.00,3.00
max,26946.37,1555.00,746.00


### Step 4: Counts for categorical variables

In [5]:
print("Recommended vs. Not Recommended:")
print(reviews["recommended"].value_counts())
print("Percent recommended:", round(reviews["recommended"].mean() * 100, 1))

print("\nSteam rating category:")
print(games["review_score"].value_counts())

print("\nFree vs. paid:")
print(games["is_free"].value_counts())

Recommended vs. Not Recommended:
recommended
True     8561
False    3284
Name: count, dtype: int64
Percent recommended: 72.3

Steam rating category:
review_score
Very Positive              38
Mixed                      38
Mostly Positive            37
Overwhelmingly Positive     6
Mostly Negative             2
Name: count, dtype: int64

Free vs. paid:
is_free
False    114
True       7
Name: count, dtype: int64


### Step 5: Statistics by genre
A game can have more than one genre, so `.explode()` gives each game one row per genre. That means one game can count toward several genres.

In [6]:
genre_df = games.assign(genre=games["genres"].str.split(", ")).explode("genre")

genre_stats = genre_df.groupby("genre").agg(
    games=("app_id", "count"),
    avg_pct_positive=("pct_positive", "mean"),
    avg_price=("price", "mean")
).sort_values("games", ascending=False).round(1)

genre_stats

,games,avg_pct_positive,avg_price
genre,,,
Indie,121,75.9,14.3
Adventure,56,75.2,15.1
Action,50,75.4,15.4
Casual,47,78.6,11.1
Simulation,43,73.2,15.4
RPG,35,73.2,16.2
Strategy,32,72.5,15.7
Early Access,14,69.0,14.9
Massively Multiplayer,8,68.4,9.4


### Step 6: Statistics by price range

In [7]:
games["price_range"] = pd.cut(
    games["price"],
    bins=[-0.01, 0, 9.99, 19.99, 29.99, 100],
    labels=["Free", "$0.01-9.99", "$10-19.99", "$20-29.99", "$30+"]
)

price_stats = games.groupby("price_range", observed=True).agg(
    games=("app_id", "count"),
    avg_pct_positive=("pct_positive", "mean"),
    median_pct_positive=("pct_positive", "median")
).round(1)

price_stats

,games,avg_pct_positive,median_pct_positive
price_range,,,
Free,7,74.7,72.4
$0.01-9.99,43,78.6,77.5
$10-19.99,55,75.8,78.1
$20-29.99,13,69.4,67.4
$30+,3,68.2,70.1


### Step 7: Positive vs. negative reviews
We use the **median** here because playtime and word count have extreme outliers (one reviewer had over 27,000 hours), which pull the mean up.

In [8]:
reviews.groupby("recommended")[["word_count", "playtime_hours"]].median().round(1)

,word_count,playtime_hours
recommended,,
False,40.0,3.4
True,23.0,7.5


### Step 8: Release year
Most of the sample is recent, which matters for our question about how reviews change after release.

In [9]:
games["release_year"].value_counts().sort_index()

,count
release_year,
2005,1
2011,1
2013,4
2014,3
2015,1
2016,3
2017,10
2018,8
2019,5
